# 9 (Capstone) - The Agent as a Trajectory Interface

Chapter 9 treats an agentic system as an **interface that returns a trajectory**, not a single answer, and lays out the testing as a sequence of stages. This companion shows that interface on the capstone: the Chapter 15 complaint agent wrapped as a `gmstest` system under test, one real trajectory read as the unit of observation, and the three-level campaign read. The per-stage *application* is Chapter 12 (notebooks 12.5-12.8); this notebook is the interface those stages drive. Like the Chapter 12 notebooks it reads **pinned artifacts** rather than re-running the agent, so it executes without a model or a store.

## The system under test is an interface

The framework drives the deployed agent **unchanged** by wrapping it as a callable that maps a customer message to a `SUTResult`. `AgentSUT` runs one message through `build_complaint_harness` and maps the resulting `Trajectory` to a `SUTResult` using the agent's own accessors, so the adapter introduces no judgment of its own: the answer is the draft, the status reflects escalation, the components carry the classification and the trajectory carries the tool order.

```python
# Reference (run separately): needs agentlab, the models and a store.
from apps.complaint_sut import AgentSUT
sut = AgentSUT()                 # wraps build_complaint_harness(), unchanged
res = sut('I was charged a $35 overdraft fee and I want it removed.')
# res.answer / res.components / res.trajectory / res.status / res.escalation_trigger
```

The two trajectories below were captured exactly this way (`scripts/capture_trajectory_example.py`) and pinned, so the notebook can read them back.

In [ ]:
import json
from pathlib import Path

# Resolve artifacts whether we run from notebooks/ or the repo root.
root = Path.cwd().parent / 'code'
while not (root / 'data' / 'capstone_run.json').exists() and root != root.parent:
    root = root.parent
TRAJ = json.loads((root / 'data' / 'capstone_trajectory_example.json').read_text())
D = json.loads((root / 'data' / 'capstone_run.json').read_text())
WORKFLOW = ['classify_complaint', 'extract_facts', 'search_policy',
            'flag_regulatory', 'draft_response']
print('loaded', len(TRAJ), 'pinned trajectories and the campaign artifact from', root / 'data')

## The trajectory is the unit of observation

A single answer conceals the path that produced it. The two runs below are both classified `complaint`, so an outcome test that recorded only the label would call them the same. They are not: one runs the full five-step workflow to a drafted reply, the other stops after the regulatory check and escalates. Reading the trajectory keeps them distinct.

In [ ]:
for t in TRAJ:
    r = t['sut_result']
    path = ' > '.join(r['tool_sequence'])
    print(f"{t['case']}: {t['message']}")
    print(f"  classification : {r['classification']}")
    print(f"  path           : {path}")
    print(f"  status         : {r['status']}"
          + (f"  (trigger: {r['escalation_trigger']})" if r['escalation_trigger'] else ''))
    if r['answer_excerpt']:
        print(f"  draft          : {r['answer_excerpt']}")
    print()

Both runs classify the message as a `complaint`, and both pass every gate -- no step is refused. There the agreement ends. `case-001` (a routine overdraft-fee dispute) runs all five tools and terminates `done` with a grounded draft; `case-004` (a mortgage-servicing grievance) runs four tools and then **escalates on the regulatory trigger**, so it never drafts a reply. The classification bit is identical; the trajectory is what tells the right outcome reached by the right route from a case that must divert to a human. The chapter's point is exactly this: reduce either run to one label and the path -- the object every testing stage scores -- is gone.

## The three-level read

With the interface in hand, a single `evaluate.run` over the composed scenarios scores every run at the three levels -- outcome, trajectory and process -- and the weak-link decomposition charges each failing run to the first tool, in workflow order, that erred.

```python
# Reference (run separately): the campaign that wrote capstone_run.json.
from gmstest import evaluate
rows = evaluate.run(scns, sut, workflow_order=WORKFLOW)
print(evaluate.summary(rows))           # n, accuracy, workflow_adherence
blame = evaluate.weak_link(rows, WORKFLOW)
```

The numbers below are read from the pinned campaign artifact, the same one the Chapter 12 notebooks use.

In [ ]:
# Operational and component read across the campaign. We report adherence, the audit
# trail and the per-tool/weak-link decomposition; the overall per-query decision is read
# separately in Chapter 12 (its attribution, not the single rate).
print(f"Scenarios            : {D['n_runs']}")
print(f"Workflow adherence   : {D['workflow_adherence']:.3f}")
print(f"Audit verifies       : {D['audit_verifies']}")
print()
print('Per-tool decomposition (label-scored; search_policy is graph-truth in 12.5)')
for name in ('classify_complaint', 'flag_regulatory'):
    t = D['per_tool'][name]
    print(f"  {name:20s} accuracy={t['accuracy']:.3f}  (scored {t['scored']})")
print()
print('Weak-link blame (wrongly-decisioned runs -> first erring tool)')
for tool, n in sorted(D['weak_link'].items(), key=lambda kv: -kv[1]):
    print(f"  {tool:20s} {n}")

Workflow adherence is perfect and the audit log verifies: every run executed the tools in the prescribed order and left a replayable record, so no failure is the agent acting out of turn. The per-tool decomposition is the component layer, and the weak-link counts attribute each wrongly-decisioned run to the first tool that erred. The overall per-query decision (right classification *and* right escalation, with the per-tool components scored separately) is deliberately absent from this read; Chapter 12 reports it alongside the **attribution** of failures rather than as a single rate on its own.

## Where each stage is tested

Chapter 9's stage table is instantiated end to end on this agent in Chapter 12, each stage in its own notebook, all reading this same pinned campaign:

| Stage | Notebook |
|---|---|
| Component (gates + per-tool + groundedness) | `12_5_component_testing` |
| System (factor + weak-link attribution) | `12_6_system_attribution` |
| Operational (trajectory + process health) | `12_7_operational` |
| Resilience (fault injection) | `12_8_resilience` |

This companion is the **interface** those stages drive; the per-stage numbers live there, so nothing is duplicated here.

## Self-check

The asserts pin the deterministic trajectory facts this notebook rests on -- two runs both classified `complaint`, one a full five-tool draft and one a four-tool regulatory escalation -- and the structural invariants of the campaign read (perfect adherence, a verifying audit, the three measured tools and a well-formed weak-link). The campaign aggregates are nondeterministic across reruns, so the checks pin their structure, not their exact values.

In [ ]:
by_case = {t['case']: t for t in TRAJ}

# Trajectory facts (deterministic, freshly captured).
c1 = by_case['case-001']['sut_result']
c4 = by_case['case-004']['sut_result']
assert c1['classification'] == 'complaint' and c4['classification'] == 'complaint'
assert c1['tool_sequence'] == WORKFLOW                # full five-step workflow
assert c1['status'] == 'done' and not c1['escalation_trigger']
assert c4['tool_sequence'] == WORKFLOW[:4]            # stops before drafting
assert c4['status'] == 'escalated' and c4['escalation_trigger'] == 'regulatory'
assert all(s['gates_not_allowed'] == [] for t in TRAJ for s in t['steps'])

# Campaign read: structural invariants (robust to nondeterministic reruns).
assert D['workflow_adherence'] == 1.0 and D['audit_verifies'] is True
assert D['n_runs'] == 120
for name in ('classify_complaint', 'search_policy', 'flag_regulatory'):
    assert 0.0 < D['per_tool'][name]['accuracy'] <= 1.0
assert D['weak_link'] and set(D['weak_link']) <= set(WORKFLOW)
assert sum(D['weak_link'].values()) > 0
print('OK - Ch9 capstone: the agent as a trajectory interface; path is the unit of observation')